# Preparación del Dataset: Tratamiento y Limpieza Inicial

Pipeline de limpieza del padrón oficial FIDE (~1.9M registros). Al final del notebook quedan dos datasets curados:

- `df_limpio`: jugadores con al menos un rating, con año de nacimiento válido y variables derivadas.
- `df_activos`: subconjunto de `df_limpio` con jugadores activos y edad ≤ 95 años.

## Índice

1. [Carga de Datos Brutos](#1-carga-de-datos-brutos)
2. [Tratamiento de Valores Nulos](#2-tratamiento-de-valores-nulos)
3. [Tipado y Optimización de Memoria](#3-tipado-y-optimización-de-memoria)
4. [Filtrado de Registros Válidos](#4-filtrado-de-registros-válidos)
   - 4.1. [Filtro por rating y año de nacimiento](#41-filtro-por-rating-y-año-de-nacimiento)
   - 4.2. [Re-imputación de títulos](#42-re-imputación-de-títulos)
   - 4.3. [Revisión de valores de `title` y `w_title`](#43-revisión-de-valores-de-title-y-w_title)
   - 4.4. [Fusión de `title` y `w_title`](#44-fusión-de-title-y-w_title)
5. [Control de Duplicados](#5-control-de-duplicados)
6. [Análisis de Flags de Actividad](#6-análisis-de-flags-de-actividad)
7. [Estadísticos Descriptivos Iniciales](#7-estadísticos-descriptivos-iniciales)
8. [Depuración de Año de Nacimiento](#8-depuración-de-año-de-nacimiento)
9. [Ingeniería de Variables](#9-ingeniería-de-variables)
   - 9.1. [Cálculo de Edad](#91-cálculo-de-edad)
   - 9.2. [Inspección de Jugadores Centenarios](#92-inspección-de-jugadores-centenarios)
   - 9.3. [Creación de Categorías por Edad](#93-creación-de-categorías-por-edad)
   - 9.4. [Creación de Perfil de Juego](#94-creación-de-perfil-de-juego)
   - 9.5. [Correcciones de títulos y género (pendiente)](#95-correcciones-de-títulos-y-género-pendiente)
10. [Segmentación de Jugadores Activos](#10-segmentación-de-jugadores-activos)
11. [Resumen de Variables Derivadas](#11-resumen-de-variables-derivadas)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

## 1. Carga de Datos Brutos

Se carga el último snapshot del archivo `fide_players_bruto.parquet` generado por el pipeline de ingesta streaming. Contiene ~1.9M registros del padrón mundial FIDE sin filtros.

In [ ]:
from src.config.settings import DATA_INTERIM_DIR
from src.utils.snapshot import latest_snapshot, interim_path

SNAPSHOT = latest_snapshot(DATA_INTERIM_DIR)
df = pd.read_parquet(interim_path(SNAPSHOT))
print(df.info())
df.head(10)

## 2. Tratamiento de Valores Nulos

- **Títulos** (`title`, `w_title`, `o_title`): los valores `NaN` se imputan como `'nt'` (no title).
- **Flag de actividad** (`flag`): los valores `NaN` se imputan como `'a'` (activo).

In [ ]:
columnas_titulos = ['title','w_title',"o_title"]

for col in columnas_titulos:
    df[col] = df[col].fillna('nt')

df['flag'] = df['flag'].fillna('a')

## 3. Tipado y Optimización de Memoria

Conversión de tipos para optimizar el uso de memoria:

- Variables numéricas (`rating`, `rapid_rating`, `blitz_rating`, `birthday`) → `Int64`
- Variables categóricas (`sex`, `country`, `flag`, títulos) → `category`
- Identificadores (`fideid`, `name`) → `string`

In [ ]:
variables_numericas = [
    'rating',
    'rapid_rating',
    'blitz_rating', 
    'birthday'
]

variables_categoricas = [
    'sex',
    'country',
    'flag',
    'w_title',
    'title',
    'o_title'
]   

for var in variables_numericas:
  df[var] = pd.to_numeric(df[var], errors="coerce").astype("Int64")

for col in variables_categoricas:
  df[col] = df[col].astype("category")

df['fideid'] = df['fideid'].astype('string')
df['name'] = df['name'].astype('string')

df.info()

Distribución de `title` y `w_title` tras el tipado, antes de filtrar.

In [ ]:
print(df['title'].value_counts())
print(df['w_title'].value_counts())

## 4. Filtrado de Registros Válidos

### 4.1. Filtro por rating y año de nacimiento

Criterios de filtrado:

- Retener solo jugadores con **al menos un rating > 0** (clásico, rápido o blitz).
- Reemplazar los ratings con valor 0 por `NaN` (indican ausencia de dato, no rating real).
- Eliminar registros sin año de nacimiento (`birthday`).

> Esto descarta ~1.1M de registros con Elo cero (jugadores escolares, árbitros, registros administrativos).
> En la última ejecución quedaron **782.834** registros.

In [ ]:
df_limpio = df[
    (df['rating'] > 0) | 
    (df['rapid_rating'] > 0) | 
    (df['blitz_rating'] > 0)
]

df_limpio = df_limpio.replace(0, np.nan)

df_limpio = df_limpio.dropna(subset=['birthday'])


df_limpio.info()

### 4.2. Re-imputación de títulos

Tras el filtrado se vuelven a imputar los títulos nulos como `'nt'` para asegurar consistencia (el `replace(0, np.nan)` anterior se aplica a todo el DataFrame).

In [ ]:
columnas_titulos = ['title','w_title',"o_title"]

for col in columnas_titulos:
    df_limpio[col] = df_limpio[col].fillna('nt')

### 4.3. Revisión de valores de `title` y `w_title`

Revisión de los valores presentes en ambas columnas antes de fusionarlas:

- Comprobación de espacios sobrantes en `title`. Nota: como `NaN != NaN` es siempre `True`, esta comparación solo es fiable si la columna ya no tiene nulos (en la última ejecución dio `0`).
- Valores únicos de `title` y de `w_title`.

In [ ]:
sin_espacios = df_limpio["title"].str.strip()
print((df_limpio["title"] != sin_espacios).sum())

In [ ]:
for valor in df_limpio["title"].dropna().unique():
    print(repr(valor))

In [ ]:
for valor in df_limpio["w_title"].dropna().unique():
    print(repr(valor))

### 4.4. Fusión de `title` y `w_title`

`w_title` solo contiene títulos femeninos (`WCM`, `WFM`, `WIM`, `WGM`). Se fusiona con `title` conservando, para cada jugador, **el título de mayor jerarquía** entre ambas columnas, y luego se elimina `w_title`.

Jerarquía utilizada (de mayor a menor):

$$GM>IM>WGM>FM>WIM>CM>WFM>WCM>nt$$

Criterio: se ordena según el rating mínimo que exige cada título; en los empates (FM = WGM y CM = WIM) el título femenino queda por encima porque exige normas.

La categórica incluye `'nt'` entre sus categorías, de modo que no se genera el `Pandas4Warning` por valores fuera de las categorías.

In [ ]:
titulos_orden = ["GM", "IM", "WGM", "FM", "WIM", "CM", "WFM", "WCM"]
orden = ["nt"] + titulos_orden[::-1]

t = pd.Categorical(df_limpio["title"], categories=orden, ordered=True)
w = pd.Categorical(df_limpio["w_title"], categories=orden, ordered=True)

codigos = np.maximum(t.codes, w.codes)
df_limpio["title"] = pd.Categorical.from_codes(codigos, categories=orden, ordered=True)
df_limpio = df_limpio.drop(columns="w_title")

print(df_limpio["title"].value_counts())

print(df_limpio["title"].value_counts().sum())

## 5. Control de Duplicados

Verificación de unicidad por `fideid`. Se espera que cada jugador tenga un ID FIDE único (en la última ejecución: 0 duplicados).

In [ ]:
duplicados_id = df_limpio.duplicated(subset=['fideid']).sum()
print(f"Cantidad de registros con FIDE ID repetido: {duplicados_id}")

if duplicados_id > 0:
    filas_repetidas = df_limpio[df_limpio.duplicated(subset=['fideid'], keep=False)]
    display(filas_repetidas)

Verificación visual del dataset tras limpieza y deduplicación.

In [ ]:
df_limpio.head()

## 6. Análisis de Flags de Actividad

Los flags de la FIDE indican el estado del jugador:

| Flag | Significado |
| :--- | :--- |
| `a` | Activo |
| `i` | Inactivo |
| `w` | Mujer activa |
| `wi` | Mujer inactiva |

Se crea la variable booleana `es_activo`, que es `False` para los flags `i` y `wi`.

In [ ]:
df_flag = df_limpio['flag'].value_counts().reset_index()
df_flag

In [ ]:
df_limpio['es_activo'] = ~df_limpio['flag'].isin(['i', 'wi'])

df_limpio.info()

## 7. Estadísticos Descriptivos Iniciales

Resumen estadístico del dataset filtrado para validar rangos y detectar anomalías.

In [ ]:
df_limpio.describe()

## 8. Depuración de Año de Nacimiento

Eliminación de registros con `birthday < 1920` por considerarse valores ilógicos o errores de registro (en la última ejecución: 1 registro).

In [ ]:
errores_nacimiento = df_limpio[df_limpio['birthday'] < 1920]
df_limpio = df_limpio[df_limpio['birthday'] >= 1920].copy()
print(f"Registros con año de nacimiento ilógico: {len(errores_nacimiento)}")

## 9. Ingeniería de Variables

### 9.1. Cálculo de Edad

Variable derivada: `edad = año_actual - birthday`.

In [ ]:
fecha = dt.datetime.now()
df_limpio['edad'] = fecha.year - df_limpio['birthday'] 

Estadísticos descriptivos tras incorporar la variable `edad`.

In [ ]:
df_limpio.describe()

### 9.2. Inspección de Jugadores Centenarios

Revisión de jugadores nacidos en 1920 (los más antiguos del dataset tras el filtro).

In [ ]:
jugador_centenario = df_limpio[df_limpio['birthday'] == 1920]
display(jugador_centenario)

### 9.3. Creación de Categorías por Edad

Clasificación en categorías de ajedrez juvenil:

| Categoría | Rango de Edad |
| :--- | :--- |
| Sub 8 | 0 – 7 años |
| Sub 10 | 8 – 9 años |
| Sub 12 | 10 – 11 años |
| Sub 14 | 12 – 13 años |
| Sub 16 | 14 – 15 años |
| Sub 18 | 16 – 17 años |
| Sub 20 | 18 – 19 años |
| Absoluta | 20+ años |

In [ ]:
orden_jerarquico = [
    "Sub 8", "Sub 10", "Sub 12", "Sub 14", 
    "Sub 16", "Sub 18", "Sub 20", "Absoluta"
]

reglas_ajedrez = {
    "Sub 8":  (df_limpio["edad"] > 0) & (df_limpio["edad"] < 8),
    "Sub 10": (df_limpio["edad"] >= 8) & (df_limpio["edad"] < 10),
    "Sub 12": (df_limpio["edad"] >= 10) & (df_limpio["edad"] < 12),
    "Sub 14": (df_limpio["edad"] >= 12) & (df_limpio["edad"] < 14),
    "Sub 16": (df_limpio["edad"] >= 14) & (df_limpio["edad"] < 16),
    "Sub 18": (df_limpio["edad"] >= 16) & (df_limpio["edad"] < 18),
    "Sub 20": (df_limpio["edad"] >= 18) & (df_limpio["edad"] < 20),
    "Absoluta": df_limpio["edad"] >= 20,
}

df_limpio["categoria"] = np.select(
    list(reglas_ajedrez.values()), 
    list(reglas_ajedrez.keys()), 
    default="Sin Categoría"
)


df_limpio['categoria'] = pd.Categorical(
    df_limpio['categoria'], 
    categories=orden_jerarquico, 
    ordered=True
)

In [ ]:
df_limpio["categoria"].value_counts()

### 9.4. Creación de Perfil de Juego

Clasifica a cada jugador según los ritmos de juego en los que tiene rating (clásico, rápido, blitz). Categorías: `completo`, `especialista_clasico`, `especialista_rapido`, `especialista_blitz`, `clasico_rapido`, `clasico_blitz`, `rapido_blitz`. Si no tiene ninguno se asigna `sin_elo`.

In [ ]:
orden = [
        "completo",
        "especialista_clasico",
        "especialista_rapido",
        "especialista_blitz",
        "clasico_rapido",
        "clasico_blitz",
        "rapido_blitz"]

has_classic = df_limpio["rating"].notna() if "rating" in df_limpio.columns else pd.Series(False, index=df_limpio.index)
has_rapid = df_limpio["rapid_rating"].notna() if "rapid_rating" in df_limpio.columns else pd.Series(False, index=df_limpio.index)
has_blitz = df_limpio["blitz_rating"].notna() if "blitz_rating" in df_limpio.columns else pd.Series(False, index=df_limpio.index)


condiciones_perfil = [
            has_classic & has_rapid & has_blitz,          # completo
            has_classic & ~has_rapid & ~has_blitz,        # especialista_clasico
            ~has_classic & has_rapid & ~has_blitz,        # especialista_rapido
            ~has_classic & ~has_rapid & has_blitz,        # especialista_blitz
            has_classic & has_rapid & ~has_blitz,         # clasico_rapido
            has_classic & ~has_rapid & has_blitz,         # clasico_blitz
            ~has_classic & has_rapid & has_blitz,         # rapido_blitz
        ]

df_limpio["player_profile"] = np.select(
            condiciones_perfil,
            orden,
            default="sin_elo"
        )

df_limpio["player_profile"] = pd.Categorical(
            df_limpio["player_profile"],
            categories=orden,
            ordered=True
        )

In [ ]:
df_limpio["player_profile"].value_counts()

### 9.5. Correcciones de títulos y género (pendiente)

> **Sección en construcción.** La celda siguiente está comentada porque, tal como estaba, no se podía ejecutar:
> - Usa la columna `w_title`, que ya se eliminó en la sección 4.4.
> - Usa `TITLE_WOMAN`, que no se importa ni se define en este notebook.
>
> La intención aparente es detectar jugadores con `sex == "M"` que tengan un título femenino.

In [ ]:
# PENDIENTE: revisar antes de ejecutar (ver nota arriba)
# df_anomalias = df_limpio[
#     (df_limpio["sex"] == "M") & 
#     (df_limpio["w_title"].isin(TITLE_WOMAN))
# ]
#
# df_anomalias

## 10. Segmentación de Jugadores Activos

Creación del subconjunto `df_activos`:

- Excluye los flags `'i'` (inactivo) y `'wi'` (mujer inactiva).
- Trunca la edad máxima a 95 años (elimina outliers inverosímiles).

En la última ejecución `df_activos` quedó con **441.904** registros.

In [ ]:
df_activos = df_limpio[~df_limpio['flag'].isin(['i', 'wi'])].copy()
df_activos = df_activos[df_activos['edad'] <= 95].copy()

df_activos.describe()

In [ ]:
df_activos.info()

## 11. Resumen de Variables Derivadas

| Variable | Sección | Descripción |
| :--- | :--- | :--- |
| `title` (fusionada) | 4.4 | Título de mayor jerarquía entre `title` y `w_title` (categórica ordenada) |
| `es_activo` | 6 | `False` si el flag es `i` o `wi` |
| `edad` | 9.1 | Año actual − `birthday` |
| `categoria` | 9.3 | Categoría por edad (Sub 8 … Absoluta) |
| `player_profile` | 9.4 | Combinación de ritmos de juego con rating |